In [24]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/My Drive/Colab Notebooks')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import numpy as np
import pandas as pd
from scipy.io.arff import loadarff


In [26]:
train_path="Data/ECG200_TRAIN.arff"
test_path="Data/ECG200_TEST.arff"
def read_ariff(path):
    raw_data, meta =loadarff(path)
    cols=[x for x in meta]
    data2d=np.zeros([raw_data.shape[0],len(cols)])
    for i,col in zip(range(len(cols)),cols):
        data2d[:,i]=raw_data[col]
    print(data2d.shape)
    return data2d

data2d=read_ariff(train_path)
test2d=read_ariff(test_path)
print(data2d.shape, test2d.shape)

(100, 97)
(100, 97)
(100, 97) (100, 97)


In [27]:
# Get the number of columns
cols = data2d.shape[1]

# Split the data
train_x = data2d[:, :cols-1]  # All columns except the last one
data_for_spike = train_x
train_y = data2d[:, cols-1]   # The last column

train_y[np.where(train_y != 1)] = 0

In [28]:
num_samples = train_x.shape[0]
split_index = int(0.7 * num_samples)

# Optional: Shuffle the data before splitting
indices = np.arange(num_samples)
np.random.shuffle(indices)

train_x = train_x[indices]
train_y = train_y[indices]

# Split
x_train = train_x[:split_index]
y_train = train_y[:split_index]

x_test = train_x[split_index:]
y_test = train_y[split_index:]

# Shapes for verification
print(x_train.shape, y_train.shape)  # e.g. (689, 136), (689,)
print(x_test.shape, y_test.shape)    # e.g. (172, 136), (172,)

(70, 96) (70,)
(30, 96) (30,)


In [29]:
train_x = x_train
test_x = x_test
train_y = y_train
test_y = y_test

Dataset reading - preprocessing

Normalizing

BaseLDN

In [30]:
import numpy as np
from scipy.linalg import expm

class BaseLDN:
    """
    A simplified Legendre Delay Network (LDN) using only NumPy.
    Implements a continuous-time delay of an input signal.
    """

    def __init__(self, order, theta, tau, dt=0.001, c2d=True):
        """
        Initialize the LDN with the given parameters.

        Args:
          order (int): Order of the system (number of states).
          theta (float): Time constant controlling delay.
          tau (float): Time constant of the low-pass filter.
          dt (float): Time step for discretization (used if c2d=True).
          c2d (bool): If True, apply continuous-to-discrete transformation.
        """
        self.order = order
        self.theta = theta
        self.tau = tau
        self.dt = dt
        self.c2d = c2d

        self._init_A_and_B_matrices()
        self._init_A_p_and_B_p_matrices()

    def _init_A_and_B_matrices(self):
        """
        Initialize continuous-time A and B matrices.
        """
        Q = np.arange(self.order)
        R = (2 * Q + 1)[:, None] / self.theta
        j, i = np.meshgrid(Q, Q)


        # Calculate continuous-time A and B matrices
        self.A = np.where(i < j, -1, (-1.0)**(i - j + 1)) * R
        self.B = (-1.0)**Q[:, None] * R


    def _init_A_p_and_B_p_matrices(self):
        """
        Initialize A' and B' matrices based on A, B, tau, and c2d.
        """
        if self.c2d:
             self.A_p = expm(self.A * self.dt)

        # Compute B_p using the formula B_p = A^(-1) (e^(A dt) - I) B
             self.B_p = np.linalg.solve(self.A, (self.A_p - np.eye(self.order)) @ self.B)
        else:
            # Continuous-time A' and B' matrices
            self.A_p = self.A * self.tau + np.eye(self.order)
            self.B_p = self.B * self.tau



    def get_transformed_matrices(self):
        """
        Returns the transformed (A' and B') matrices.
        """
        return self.A_p, self.B_p


In [31]:
# Define parameters
order = 8     # Order of the system
theta = 0.5   # Time constant for delay
tau = 0.1     # Low-pass filter time constant
dt = 0.1    # Time step for discretization

# Instantiate the BaseLDN class with continuous-to-discrete transformation
ldn_discrete = BaseLDN(order=order, theta=theta, tau=tau, dt=dt, c2d=True)

# Get and print the discrete-time A' and B' matrices
A_p_disc, B_p_disc = ldn_discrete.get_transformed_matrices()
np.set_printoptions(precision=4, suppress=True)  # Set formatting for readability

print("Discrete-time A':\n", A_p_disc)
print("Discrete-time B':\n", B_p_disc)


Discrete-time A':
 [[ 0.797  -0.1569 -0.0992 -0.0287  0.0084  0.0287  0.0179 -0.002 ]
 [ 0.4708  0.4254 -0.3746 -0.1322  0.006   0.0836  0.0607 -0.0021]
 [-0.496   0.6243  0.0616 -0.4153 -0.078   0.1188  0.1223  0.0109]
 [ 0.2009 -0.3085  0.5815 -0.1689 -0.4014  0.0676  0.1998  0.0472]
 [ 0.0754 -0.018  -0.1404  0.5161 -0.3768 -0.2781  0.2343  0.1111]
 [-0.316   0.3065 -0.2613  0.1062  0.3399 -0.505  -0.0154  0.1645]
 [ 0.2324 -0.2632  0.3181 -0.3711  0.3385  0.0182 -0.3311  0.0046]
 [ 0.0302 -0.0103 -0.0326  0.1011 -0.1852  0.2244 -0.0053 -0.1113]]
Discrete-time B':
 [[ 0.203 ]
 [-0.4708]
 [ 0.496 ]
 [-0.2009]
 [-0.0754]
 [ 0.316 ]
 [-0.2324]
 [-0.0302]]


pytorch LDN

In [32]:
import torch
import numpy as np

class PyTorchLDN(BaseLDN):
    def __init__(self, order, theta, tau, dt=0.1, c2d=True):
        """
        Initialize the PyTorch-based LDN with the given parameters.

        Args:
          order (int): Order of the system (number of states).
          theta (float): Time constant controlling delay.
          tau (float): Time constant of the low-pass filter.
          dt (float): Time step for discretization (used if c2d=True).
          c2d (bool): If True, apply continuous-to-discrete transformation.
        """
        # Initialize the BaseLDN (with A and B matrices)
        super().__init__(order, theta, tau, dt, c2d)

        # Get transformed A' and B' matrices from BaseLDN
        self.A_p, self.B_p = self.get_transformed_matrices()

        # Convert A' and B' matrices to Torch tensors and set dtype to float32
        self.A_p = torch.from_numpy(self.A_p).float()
        self.B_p = torch.from_numpy(self.B_p).float()

    def get_ldn_transform(self, u_t, m):
        """
        Perform the LDN transformation.

        Args:
            u_t <torch.Tensor>: Input scalar (1x1 tensor).
            m <torch.Tensor>: State-vector tensor.
        """
        # Ensure u_t and m have the same dtype as A_p and B_p
        u_t = u_t.float()
        m = m.float()

        return torch.mm(self.A_p, m) + torch.mm(self.B_p, u_t)

    def get_ldn_sigs(self, u):
        """
        Compute LDN signals using matrix multiplications.

        Args:
            u <torch.Tensor>: The input tensor (1xN).
        """
        if isinstance(u, np.ndarray):
            u = torch.from_numpy(u)

        # Ensure input tensor is float32
        u = u.float()

        m = torch.zeros(self.order, 1).float()
        out = torch.zeros(u.shape[1], self.order).float()

        for i, u_t in enumerate(u[0]):
            u_t = u_t.reshape(1, 1)
            out[i, :] = self.get_ldn_transform(u_t, m).reshape(-1)
            m[:, 0] = out[i, :]  # Update the state vector

        return out


In [33]:
# Example parameters
order = 8
theta = 0.5
tau = 0.1
u = np.array([[1.0, 0.5, 0.3, 0.8]])  # Example input signal

# Create an instance of PyTorchLDN
ldn = PyTorchLDN(order, theta, tau)

# Compute the LDN signals
ldn_signals = ldn.get_ldn_sigs(u)

# Print the LDN signals
print("LDN Signals:\n", ldn_signals)


LDN Signals:
 tensor([[ 2.0296e-01, -4.7084e-01,  4.9601e-01, -2.0090e-01, -7.5416e-02,
          3.1601e-01, -2.3243e-01, -3.0239e-02],
        [ 2.9806e-01, -4.8746e-01, -1.7942e-02,  4.1166e-01, -3.0455e-01,
         -3.8799e-01,  3.4422e-01,  4.8871e-02],
        [ 3.5728e-01, -2.6945e-01, -4.5510e-01,  2.3714e-01,  5.3236e-01,
         -5.2518e-03, -2.5449e-01,  9.2969e-03],
        [ 5.2750e-01, -1.9669e-01, -1.4831e-01, -5.7496e-01, -1.0002e-01,
          3.9047e-01, -3.7341e-04, -7.1245e-02]])


In [34]:
u = train_x[0:1, :]  # Use the first row of x_train as input
u.shape

(1, 96)

In [35]:
# Assuming x_train is already defined and available
u = train_x[0:1, :]  # Use the first row of x_train as input
u.shape
# Create an instance of PyTorchLDN
ldn = PyTorchLDN(order, theta, tau)

# Compute the LDN signals
ldn_signals = ldn.get_ldn_sigs(u)

# Print the LDN signals
print("LDN Signals:\n", ldn_signals)


LDN Signals:
 tensor([[ 1.1505e-01, -2.6691e-01,  2.8118e-01, -1.1389e-01, -4.2752e-02,
          1.7914e-01, -1.3176e-01, -1.7142e-02],
        [ 2.3174e-01, -4.2198e-01,  1.4326e-01,  1.7122e-01, -1.9597e-01,
         -1.2220e-01,  1.2324e-01,  1.8350e-02],
        [ 6.6530e-01, -1.1632e+00,  6.4210e-01, -1.0505e-01,  6.9887e-02,
          4.5399e-01, -4.6216e-01, -3.4824e-02],
        [ 1.3278e+00, -1.9525e+00,  6.5641e-01,  1.2884e-01, -5.8705e-01,
          9.3636e-02,  1.2126e-01, -4.0046e-03],
        [ 1.6527e+00, -1.2845e+00, -9.4672e-01,  1.1418e+00,  2.0015e-01,
         -8.6962e-01,  3.3762e-01,  1.2836e-01],
        [ 1.5402e+00,  4.3570e-01, -2.2847e+00, -5.8580e-02,  1.1380e+00,
         -5.8919e-02, -3.6420e-02, -3.5343e-02],
        [ 1.1108e+00,  2.4327e+00, -1.4031e+00, -1.3321e+00,  7.9449e-02,
          2.0664e-01,  2.5959e-01, -6.6964e-02],
        [ 5.7070e-01,  2.5754e+00,  1.1865e+00, -9.6697e-01, -4.3931e-01,
          3.3802e-01, -2.5034e-01, -2.4762e-02],
  

BASIC SPiking neuron

Encoder layer

In [36]:
import torch
from abc import ABC, abstractmethod

class BaseSpikingNeuron(ABC):
    def __init__(self, v_threshold=1, gain=1, bias=0, dtype=torch.float32):
        self._dtype = dtype  # Set the data type manually
        self.v = torch.tensor(0.0, dtype=dtype)  # Membrane potential
        self.v_threshold = torch.tensor(v_threshold, dtype=dtype)
        self.gain = torch.tensor(gain, dtype=dtype)
        self.bias = torch.tensor(bias, dtype=dtype)

    def spike_and_reset(self):
        """
        Checks if the neuron spikes (membrane potential >= threshold) and resets voltage.
        Returns:
          spike (Tensor): A boolean tensor indicating the spike occurrence.
        """
        spike = self.v >= self.v_threshold  # Create boolean tensor for spike
        if torch.any(spike):  # If any spikes occurred
            self.v[spike] = 0.0  # Reset membrane potential for spiking neurons
        return spike.int()

    def reset_v(self):
        """Resets membrane potential to zero."""
        self.v = torch.tensor(0.0, dtype=self._dtype)

    @abstractmethod
    def encode(self, x):
        """Encodes input x. Needs to be implemented in derived class."""
        pass

class Encoder(BaseSpikingNeuron):
    def __init__(self, encoder=1, v_threshold=1, gain=1, bias=0.5, dtype=torch.float32):
        super().__init__(v_threshold=v_threshold, gain=gain, bias=bias, dtype=dtype)
        self.encoder_value = torch.as_tensor(encoder, dtype=self._dtype)
        # Initialize self.v with a shape of 16, or based on the input size you expect
        self.v = None  # Assuming 16 neurons or a vector of length 16

    def encode(self, x):
        """Encodes the input vector `x` into spikes using IF spiking mechanism."""
        x = torch.as_tensor(x, dtype=self._dtype)
        if self.v is None or self.v.shape != x.shape:
            self.v = torch.zeros_like(x, dtype=self._dtype)

        # Ensure self.gain, self.encoder_value, and self.bias have shapes compatible with x
        if self.gain.dim() == 0:  # If self.gain is scalar, expand it to match x
            self.gain = self.gain.expand_as(x)
        if self.encoder_value.dim() == 0:  # If self.encoder_value is scalar, expand it to match x
            self.encoder_value = self.encoder_value.expand_as(x)
        if self.bias.dim() == 0:  # If self.bias is scalar, expand it to match x
            self.bias = self.bias.expand_as(x)

        # Compute the input current (J) based on gain, encoder, and bias.
        current_input = self.gain * self.encoder_value * x + self.bias

        # Ensure current_input matches the shape of self.v
        if current_input.shape != self.v.shape:
            raise RuntimeError(f"Shape mismatch: current_input has shape {current_input.shape} but self.v has shape {self.v.shape}")

        # Update the membrane potential (v) by adding current.
        print(self.v)
        self.v += current_input


        # Ensure membrane potential does not go negative (rectification).
        self.v[self.v < 0] = 0.0

        # Check for spikes and reset membrane potential where spike occurs.
        spike = self.spike_and_reset()

        return spike


'''
    def encode(self, x):
        """Encodes the input vector `x` into spikes using IF spiking mechanism."""
        x = torch.as_tensor(x, dtype=self._dtype)

        # Compute the input current (J) based on gain, encoder, and bias.
        current_input = self.gain * self.encoder_value * x + self.bias

        # Update the membrane potential (v) by adding current.
        self.v += current_input

        # Ensure membrane potential does not go negative (rectification).
        self.v[self.v < 0] = 0.0

        # Check for spikes and reset membrane potential where spike occurs.
        spike = self.spike_and_reset()

        return spike
'''


class EncoderLayer(Encoder):
    def __init__(self, order, num_neurons, v_threshold=1, gain=1, bias=0.5, dtype=torch.float32):
        super().__init__(v_threshold=v_threshold, gain=gain, bias=bias, dtype=dtype)
        self.order = order
        self.num_neurons = num_neurons  # Should be 2 * order (for duplication)

    def duplicate_input(self, x):
        """Duplicates each element in the input tensor `x` to create a new tensor of size 2d."""
        x_dup = torch.repeat_interleave(x, 2)  # Duplicate each element in x
        return x_dup

    def create_alternating_encoder_values(self, length):
        """Creates alternating encoder values [1, -1, 1, -1, ...] of size `length`."""
        encoder_values = torch.ones(length, dtype=self._dtype)
        encoder_values[1::2] = -1  # Set every second element to -1
        return encoder_values

    def encode_layer(self, x):
        """Encodes the duplicated input `x` using the Encoder mechanism with alternating encoder values."""
        # Duplicate the input tensor (convert order-length tensor to 2*order-length tensor)
        x_dup = self.duplicate_input(x)

        # Set encoder values to alternate between 1 and -1 for 2*order length
        self.encoder_value = self.create_alternating_encoder_values(x_dup.shape[0])

        # Use the inherited encode function to process the duplicated input
        spikes = self.encode(x_dup)

        return spikes


In [37]:
import torch

# Given parameters
order = 10
v_threshold = 1.0
bias = 0
gain = 1.0
num_neurons = 2 * order  # 16 neurons
timesteps = train_x.shape[1]   # Assuming there are 152 timesteps
theta = 0.5
tau = 0.1
# Create an instance of EncoderLayer
encoder = EncoderLayer(order=order, num_neurons=num_neurons, v_threshold=v_threshold, gain=gain, bias=bias)

# Initialize a list to hold the stacked spikes for all samples
stacked_spikes = []

# Process each sample in train_x
for i in range(train_x.shape[0]):  # Loop over all samples
    first_sample = train_x[i, :].reshape(1, -1)  # Reshape to match input format

    # Create an instance of PyTorchLDN for each sample
    ldn = PyTorchLDN(order, theta, tau)

    # Compute the LDN signals for the current sample
    ldn_first_sample_signals = ldn.get_ldn_sigs(first_sample)

    # Initialize a list to hold spikes for the current sample
    current_sample_spikes = []

    # Encoding over 152 timesteps using LDN output as input
    for t in range(timesteps):
        x_t = ldn_first_sample_signals[t]  # Extract the input for timestep `t` (size `order`)
        spikes = encoder.encode_layer(x_t)  # Process input and generate spikes
        current_sample_spikes.append(spikes)  # Collect spikes for current sample

    # Stack the spikes for the current sample
    current_sample_spikes_tensor = torch.stack(current_sample_spikes)  # Shape will be [152, 16]


    # Transpose to get shape [16, 152]
    combined_output = current_sample_spikes_tensor.transpose(0, 1)  # Shape will be [16, 152]

    # Append the combined output for the current sample
    stacked_spikes.append(combined_output)

# Stack all samples into a single tensor
final_output = torch.stack(stacked_spikes)  # Shape will be [6164, 16, 152]

# Print the final output shape
print("Final output shape:", final_output.shape)  # Should be [6164, 16, 152]


Streaming output truncated to the last 5000 lines.
        0.5675, 0.3531, 0.3510, 0.4006, 0.3967, 0.6865, 0.3098, 0.3441, 0.6135,
        0.4045, 0.2191])
tensor([0.2226, 0.1736, 0.5602, 0.0027, 0.3372, 0.0833, 0.6342, 0.3190, 0.3003,
        0.4920, 0.4551, 0.2490, 0.3455, 0.4518, 0.6716, 0.3247, 0.3916, 0.5661,
        0.3871, 0.2364])
tensor([0.1914, 0.2047, 0.5730, 0.0000, 0.3961, 0.0245, 0.5669, 0.3862, 0.1878,
        0.6045, 0.4636, 0.2405, 0.4190, 0.3783, 0.6968, 0.2995, 0.3630, 0.5947,
        0.3799, 0.2436])
tensor([0.1571, 0.2390, 0.5677, 0.0053, 0.4612, 0.0000, 0.6713, 0.2818, 0.2052,
        0.5871, 0.4047, 0.2994, 0.3843, 0.4130, 0.6921, 0.3042, 0.3696, 0.5881,
        0.3864, 0.2371])
tensor([0.1100, 0.2861, 0.5249, 0.0481, 0.4010, 0.0602, 0.6769, 0.2762, 0.2522,
        0.5401, 0.4303, 0.2738, 0.3874, 0.4099, 0.6846, 0.3118, 0.3653, 0.5924,
        0.3888, 0.2347])
tensor([0.0916, 0.3045, 0.4945, 0.0785, 0.4033, 0.0579, 0.6193, 0.3339, 0.2299,
        0.5624, 0.4550, 

In [38]:
import torch

# Assuming final_output is your stacked tensor of 0s and 1s
percentage_ones = (final_output.sum().item() / final_output.numel()) * 100

print(f"Percentage of 1s: {percentage_ones:.2f}%")

Percentage of 1s: 4.06%


In [39]:
import torch
import torch.nn as nn
import torch.optim as optim


d = order
n = train_x.shape[1]
input_dim = 2 * d * n  # Flattened input size (16 * 152)


#train_x = torch.randn(batch_size, 2 * d, n)  # Shape: (999, 16, 152)
x_train = final_output.float()
#train_y = torch.randint(0, 2, (batch_size,))  # Binary labels for two classes
# Convert train_y from numpy array to torch tensor
train_y_tensor = torch.tensor(train_y, dtype=torch.long)  # Ensure the correct dtype for classification

# Define the model
class Classifier(nn.Module):
    def __init__(self, input_dim):
        super(Classifier, self).__init__()
        self.fc = nn.Linear(input_dim, 2)  # Fully connected layer for 2 classes

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten 2*d x n to (batch_size, 2*d*n)
        return self.fc(x)

# Instantiate the model
model = Classifier(input_dim)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.05, weight_decay=0.01)

# Training loop
num_epochs = 40
for epoch in range(num_epochs):
    model.train()

    # Forward pass
    outputs = model(x_train)
    loss = criterion(outputs, train_y_tensor)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [1/40], Loss: 0.7116
Epoch [2/40], Loss: 0.9432
Epoch [3/40], Loss: 0.2695
Epoch [4/40], Loss: 0.2587
Epoch [5/40], Loss: 0.3059
Epoch [6/40], Loss: 0.1812
Epoch [7/40], Loss: 0.0597
Epoch [8/40], Loss: 0.0222
Epoch [9/40], Loss: 0.0310
Epoch [10/40], Loss: 0.0471
Epoch [11/40], Loss: 0.0520
Epoch [12/40], Loss: 0.0419
Epoch [13/40], Loss: 0.0254
Epoch [14/40], Loss: 0.0127
Epoch [15/40], Loss: 0.0065
Epoch [16/40], Loss: 0.0040
Epoch [17/40], Loss: 0.0032
Epoch [18/40], Loss: 0.0034
Epoch [19/40], Loss: 0.0042
Epoch [20/40], Loss: 0.0054
Epoch [21/40], Loss: 0.0067
Epoch [22/40], Loss: 0.0078
Epoch [23/40], Loss: 0.0085
Epoch [24/40], Loss: 0.0087
Epoch [25/40], Loss: 0.0085
Epoch [26/40], Loss: 0.0080
Epoch [27/40], Loss: 0.0076
Epoch [28/40], Loss: 0.0076
Epoch [29/40], Loss: 0.0080
Epoch [30/40], Loss: 0.0089
Epoch [31/40], Loss: 0.0103
Epoch [32/40], Loss: 0.0120
Epoch [33/40], Loss: 0.0137
Epoch [34/40], Loss: 0.0150
Epoch [35/40], Loss: 0.0159
Epoch [36/40], Loss: 0.0165
E

In [40]:
train_x.shape[1]

96

In [41]:


print(final_output.shape)  # Should be (999, 16, 152)
print(train_y_tensor.shape)  # Should be (999,)


torch.Size([70, 20, 96])
torch.Size([70])


In [42]:
# Switch model to evaluation mode
model.eval()
with torch.no_grad():
    # Ensure final_output is of float type
    final_output = final_output.float()

    # Perform inference
    outputs = model(final_output)  # Get the predictions
    _, predicted = torch.max(outputs, 1)  # Get the predicted class

    # Calculate accuracy
    accuracy = (predicted == train_y_tensor).sum().item() / train_y_tensor.size(0)
    print(f'Accuracy: {accuracy * 100:.2f}%')


Accuracy: 100.00%


In [43]:
import torch

# Given parameters
order = 10
v_threshold = 1.0
bias = 0
gain = 1.0
num_neurons = 2 * order  # 16 neurons
timesteps = train_x.shape[1]   # Assuming there are 152 timesteps
theta = 0.5
tau = 0.1
# Create an instance of EncoderLayer
encoder = EncoderLayer(order=order, num_neurons=num_neurons, v_threshold=v_threshold, gain=gain, bias=bias)

# Initialize a list to hold the stacked spikes for all samples
stacked_spikes = []

# Process each sample in train_x
for i in range(test_x.shape[0]):  # Loop over all samples
    first_sample = test_x[i, :].reshape(1, -1)  # Reshape to match input format

    # Create an instance of PyTorchLDN for each sample
    ldn = PyTorchLDN(order, theta, tau)

    # Compute the LDN signals for the current sample
    ldn_first_sample_signals = ldn.get_ldn_sigs(first_sample)

    # Initialize a list to hold spikes for the current sample
    current_sample_spikes = []

    # Encoding over 152 timesteps using LDN output as input
    for t in range(timesteps):
        x_t = ldn_first_sample_signals[t]  # Extract the input for timestep `t` (size `order`)
        spikes = encoder.encode_layer(x_t)  # Process input and generate spikes
        current_sample_spikes.append(spikes)  # Collect spikes for current sample

    # Stack the spikes for the current sample
    current_sample_spikes_tensor = torch.stack(current_sample_spikes)  # Shape will be [152, 16]


    # Transpose to get shape [16, 152]
    combined_output = current_sample_spikes_tensor.transpose(0, 1)  # Shape will be [16, 152]

    # Append the combined output for the current sample
    stacked_spikes.append(combined_output)

# Stack all samples into a single tensor
final_output = torch.stack(stacked_spikes)  # Shape will be [6164, 16, 152]

# Print the final output shape
print("Final output shape:", final_output.shape)  # Should be [6164, 16, 152]


Streaming output truncated to the last 5000 lines.
        0.4027, 0.3174])
tensor([0.4753, 0.0000, 0.3359, 0.0000, 0.1027, 0.0952, 0.4111, 0.4219, 0.4735,
        0.5249, 0.4213, 0.1455, 0.5123, 0.4525, 0.6224, 0.2964, 0.2766, 0.4180,
        0.4198, 0.3002])
tensor([0.9297, 0.0000, 0.3969, 0.0000, 0.1579, 0.0400, 0.3818, 0.4512, 0.5095,
        0.4888, 0.4544, 0.1125, 0.4302, 0.5345, 0.6129, 0.3059, 0.3338, 0.3608,
        0.4045, 0.3156])
tensor([0.0000, 0.0000, 0.3852, 0.0116, 0.2178, 0.0000, 0.3631, 0.4699, 0.4444,
        0.5540, 0.4770, 0.0899, 0.5020, 0.4628, 0.6308, 0.2880, 0.3003, 0.3943,
        0.3994, 0.3206])
tensor([0.4326, 0.0000, 0.3988, 0.0000, 0.2121, 0.0057, 0.4655, 0.3675, 0.4720,
        0.5264, 0.4019, 0.1649, 0.4875, 0.4773, 0.6229, 0.2958, 0.2893, 0.4053,
        0.4169, 0.3031])
tensor([0.8168, 0.0000, 0.4866, 0.0000, 0.0646, 0.1532, 0.4669, 0.3661, 0.5498,
        0.4486, 0.3973, 0.1696, 0.4963, 0.4684, 0.6020, 0.3167, 0.2685, 0.4261,
        0.4379, 0.2822])

In [44]:
test_y_tensor = torch.tensor(test_y, dtype=torch.long)
# Switch model to evaluation mode
model.eval()
with torch.no_grad():
    # Ensure final_output is of float type
    final_output = final_output.float()

    # Perform inference
    outputs = model(final_output)  # Get the predictions
    _, predicted = torch.max(outputs, 1)  # Get the predicted class

    # Calculate accuracy
    accuracy = (predicted == test_y_tensor).sum().item() / test_y_tensor.size(0)
    print(f'Accuracy: {accuracy * 100:.2f}%')


Accuracy: 86.67%
